# How to view extracted DICOM headers for MIDRC images
---
**Author:** Christopher Meyer, PhD; Assistant Director of User Solutions and Scientific Support.<br>**Last updated**: January 28, 2026<br>

**Overview:** This notebook demonstrates how to access and review the DICOM headers for imaging studies / series in the MIDRC data commons. 

* First, a query will be used to select a subset of imaging series.
* Next, the DICOM headers for those imaging series will be accessed using the property `dicom_header_file_guid` in the data_file (or imaging_study) index.
* Finally, the downloaded DICOM header files will be concatenated into a dataframe for exploratory analysis.

## 1) Set up Python environment
---


### Download an API key file containing your credentials
---

1) Navigate to the MIDRC data portal in your browser: https://data.midrc.org.

2) Read and accept the DUA (if you haven't already).

3) Navigate to the user profile page: https://data.midrc.org/identity

4) Click on the button "Create API Key" and save the `credentials.json` file somewhere safe as "midrc-credentials.json".


### Set local variables
---
Change the following `cred` variable path to point to your credentials file downloaded from the MIDRC data portal following the instructions above.

In [1]:
# ## The MIDRC Data Commons (production environment)
# cred = "/Users/cgmeyer/Downloads/midrc-credentials.json" # location of your MIDRC credentials, downloaded from https://data.midrc.org/identity by clicking "Create API key" button and saving the credentials.json locally
# api = "https://data.midrc.org" # The base URL of the data commons being queried. This shouldn't change for MIDRC.


In [ ]:
## MIDRC Staging Environment
scred = "/Users/cgmeyer/Downloads/midrc-staging-credentials.json" # location of your MIDRC credentials, downloaded from https://data.midrc.org/identity by clicking "Create API key" button and saving the credentials.json locally
sapi = "https://staging.midrc.org" # The base URL of the data commons being queried. This shouldn't change for MIDRC.


### Install / Import Python Packages and Scripts

In [2]:
## The packages below may be necessary for users to install according to the imports necessary in the subsequent cells.

import sys
#!{sys.executable} -m pip install
#!{sys.executable} -m pip install --upgrade pandas
#!{sys.executable} -m pip install --upgrade --ignore-installed PyYAML
#!{sys.executable} -m pip install --upgrade pip
# !{sys.executable} -m pip install --upgrade gen3
# !{sys.executable} -m pip install pydicom
#!{sys.executable} -m pip install --upgrade Pillow
#!{sys.executable} -m pip install psmpy
#!{sys.executable} -m pip install python-gdcm --upgrade
#!{sys.executable} -m pip install pylibjpeg --upgrade

In [3]:
## Import Python Packages and scripts

import os, subprocess
import pandas as pd
import numpy as np
import pydicom
from PIL import Image
import glob
import re
from pathlib import Path

# import some Gen3 packages
import gen3
from gen3.auth import Gen3Auth
from gen3.query import Gen3Query



### Initiate instances of the Gen3 SDK Classes using credentials file for authentication
---
Again, make sure the "cred" directory path variable reflects the location of your credentials file (path variables set above).

In [4]:
# ## The MIDRC Data Commons (production environment)
# auth = Gen3Auth(api, refresh_file=cred) # authentication class
# query = Gen3Query(auth) # query class


In [5]:
## MIDRC Staging Environment
sauth = Gen3Auth(sapi, refresh_file=scred) # authentication class
squery = Gen3Query(sauth) # query class


## 2) Build Cohorts by Sending Queries to the MIDRC APIs
#### General notes on sending queries:
* There are many ways to query and access metadata for cohort building in MIDRC, but this notebook will focus on using the [Gen3](https://gen3.org) graphQL query service ["guppy"](https://github.com/uc-cdis/guppy/#readme). This is the backend query service that [MIDRC's data explorer GUI](https://data.midrc.org/explorer) uses. So, anything you can do in the explorer GUI, you can do with guppy queries, and more!
* The guppy graphQL service has more functionality than is demonstrated in this simple example. You can find extensive documentation in GitHub [here](https://github.com/uc-cdis/guppy/blob/master/doc/queries.md) in case you'd like to build your own queries from scratch.
* The Gen3 SDK (intialized as `query` above in this notebook) has Python wrapper scripts to make sending queries to the guppy graphQL API simpler. The guppy SDK package can be viewed in GitHub [here](https://github.com/uc-cdis/gen3sdk-python/blob/master/gen3/query.py).
* Guppy queries focus on a particular type of data (cases, imaging studies, files, etc.), which corresponds to the major tabs in [MIDRC's data explorer GUI](https://data.midrc.org/explorer).
* Queries include arguments that are akin to selecting filter values in [MIDRC's data explorer GUI](https://data.midrc.org/explorer).
* To see more documentation about how to use and combine filters with various operator logic (like AND/OR/IN, etc.) see [this page](https://github.com/uc-cdis/guppy/blob/master/doc/queries.md#filter).

---


#### Set query parameters
---
* Here, we'll send a query to the `imaging_study` guppy index, which corresponds to the "Imaging Studies" tab of [MIDRC's data explorer GUI](https://data.midrc.org/explorer).
* The filters defined below can be modified to return different subsets of imaging studies. Here, we'll use rather restrictive parameters so the number of studies returned is small for demonstration purposes.
* If our query request is successful, the API response should be in JSON format, and it should contain a list of imaging study UIDs along with any other study-related data we ask for.


In [6]:
### Set some "imaging_study" query parameters

## Project ID: We're interested only in a particular dataset in MIDRC
project_id = "TCIA-RICORD_1a"

## Imaging study modality filter: we select only imaging studies with a modality of CT
study_modalities = ["CT"]

## Imaging study body part filter: here we select "chest" as the "LOINC system" filter, which is the body part examined
body_part_examined = "Chest"

## Case filters: we will select Hispanic males 70 years of age and older
sex = "Male"
age_threshold = 40

In [7]:
## Note: the "fields" option defines what fields we want the query to return. If set to "None", returns all available fields.

imaging_studies = squery.raw_data_download(
                    data_type="imaging_study",
                    fields=None,
                    filter_object={
                        "AND": [
                            {"=": {"project_id": project_id}},
                            {"=": {"loinc_system": body_part_examined}},
                            {"=": {"sex": sex}},
                            {">=": {"age_at_index": age_threshold}},
                            {"IN": {"study_modality": study_modalities}},
                        ]
                    },
                    sort_fields=[{"submitter_id": "asc"}]
                )

if len(imaging_studies) > 0:
    imaging_studies_ids = [i['submitter_id'] for i in imaging_studies if 'submitter_id' in i] ## make a list of the imaging study IDs returned
    print("Query returned {} study IDs from {} cases.".format(len(imaging_studies),len(set([i['case_ids'][0] for i in imaging_studies if 'case_ids' in i]))))
    print("Data is a list with rows like this:\n\t {}".format(imaging_studies[0:1]))
else:
    print("Your query returned no data! Please, check that query parameters are valid.")

Query returned 21 study IDs from 17 cases.
Data is a list with rows like this:
	 [{'_imaging_study_id': '15518781-2e0b-4f0d-a16b-60248b8e09af', 'project_id': 'TCIA-RICORD_1a', 'submitter_id': '1.2.826.0.1.3680043.10.474.419639.152757649181548418723364589755', 'case_ids': ['419639-001012'], 'age_at_imaging': 79, 'loinc_code': '29252-4', 'loinc_contrast': 'WO', 'loinc_long_common_name': 'CT Chest WO contrast', 'loinc_method': 'CT', 'loinc_system': 'Chest', 'study_description': 'CT CHEST WITHOUT CONTRAST', 'study_modality': ['CT'], 'study_year': 2004, 'study_year_shifted': 'true', 'study_uid': '1.2.826.0.1.3680043.10.474.419639.152757649181548418723364589755', 'sex': ['Male'], 'race': [], 'age_at_index': [79], 'index_event': [], 'zip': [], 'covid19_positive': ['Yes'], 'ethnicity': [], 'dataset_submitter_id': ['RICORD_1a'], 'license': ['https://creativecommons.org/licenses/by-nc/4.0/'], 'data_url_doi': ['https://doi.org/10.7937/VTW4-X588'], 'data_contributor': ['TCIA'], '_mr_series_file_co

In [8]:
imaging_studies_df = pd.DataFrame(imaging_studies)
display(imaging_studies_df)

,_imaging_study_id,project_id,submitter_id,case_ids,age_at_imaging,loinc_code,loinc_contrast,loinc_long_common_name,loinc_method,loinc_system,...,data_category,dicom_header_file_guid,data_file_source_node,data_file_annotation_name,data_file_annotation_method,data_file_image_data_modified,data_file_image_data_modification_name,data_file_image_data_modification_method,auth_resource_path,body_part_examined
0,15518781-2e0b-4f0d-a16b-60248b8e09af,TCIA-RICORD_1a,1.2.826.0.1.3680043.10.474.419639.152757649181...,[419639-001012],79,29252-4,WO,CT Chest WO contrast,CT,Chest,...,[CT],"[dg.MD1R/24b4eb04-48bd-47ca-802d-4c7441318e40,...",[ct_series_file],[],[],[],[],[],/programs/TCIA/projects/RICORD_1a,NaN
1,9da06350-31b8-485b-932e-fcee2e6db243,TCIA-RICORD_1a,1.2.826.0.1.3680043.10.474.419639.198191490979...,[419639-001476],79,24628-0,W,CT Chest W contrast IV,CT,Chest,...,[CT],"[dg.MD1R/bb1ea127-bc16-4d5e-8766-9ca91d313ef3,...",[ct_series_file],[],[],[],[],[],/programs/TCIA/projects/RICORD_1a,NaN
2,e89fdddd-176f-4ccb-8553-de1d31a39733,TCIA-RICORD_1a,1.2.826.0.1.3680043.10.474.440808.3879,[440808-000020],70,79077-4,W,CTA Pulmonary arteries for pulmonary embolus W...,CT.angio,Chest,...,[CT],[dg.MD1R/7f9d9a20-86f5-4f7b-9395-b1fedfc3f210],[ct_series_file],[],[],[],[],[],/programs/TCIA/projects/RICORD_1a,NaN
3,1ce75c3c-a0ef-45d6-938c-68be431d8cce,TCIA-RICORD_1a,1.2.826.0.1.3680043.10.474.440808.929,[440808-000011],65,79077-4,W,CTA Pulmonary arteries for pulmonary embolus W...,CT.angio,Chest,...,[CT],[dg.MD1R/0890903c-2078-4d38-8d1b-b9c6c1a6cb61],[ct_series_file],[],[],[],[],[],/programs/TCIA/projects/RICORD_1a,[CHEST]
4,5cd05fa2-cfa1-48e0-a4ab-cd057494827b,TCIA-RICORD_1a,1.2.826.0.1.3680043.10.474.440808.1782,[440808-000012],55,79077-4,W,CTA Pulmonary arteries for pulmonary embolus W...,CT.angio,Chest,...,[CT],[dg.MD1R/7521005c-79b2-4ce3-b157-e65a880244ad],[ct_series_file],[],[],[],[],[],/programs/TCIA/projects/RICORD_1a,[CHEST]
5,ada2af8e-5be3-4f83-8594-3638d442e13d,TCIA-RICORD_1a,1.2.826.0.1.3680043.10.474.440808.811,[440808-000016],60,79077-4,W,CTA Pulmonary arteries for pulmonary embolus W...,CT.angio,Chest,...,[CT],[dg.MD1R/a47f1e42-478d-4687-bbc9-04286ad82f18],[ct_series_file],[],[],[],[],[],/programs/TCIA/projects/RICORD_1a,[CHEST]
6,c808d05c-6ae1-47a5-b1e5-5927121c0e5e,TCIA-RICORD_1a,1.2.826.0.1.3680043.10.474.419639.290296058041...,[419639-000906],62,29252-4,WO,CT Chest WO contrast,CT,Chest,...,[CT],"[dg.MD1R/4703d586-75d9-464a-a1fa-89e259128b63,...",[ct_series_file],[],[],[],[],[],/programs/TCIA/projects/RICORD_1a,NaN
7,e28e3815-0419-4f8e-9d6f-47b776d512f2,TCIA-RICORD_1a,1.2.826.0.1.3680043.10.474.440808.1138,[440808-000012],55,79077-4,W,CTA Pulmonary arteries for pulmonary embolus W...,CT.angio,Chest,...,[CT],[dg.MD1R/385ac79c-3d20-40c8-8cec-e112c372ca1f],[ct_series_file],[],[],[],[],[],/programs/TCIA/projects/RICORD_1a,[CHEST]
8,3a79f35e-b563-446d-9f2f-be75c547d56c,TCIA-RICORD_1a,1.2.826.0.1.3680043.10.474.440808.3754,[440808-000018],55,79077-4,W,CTA Pulmonary arteries for pulmonary embolus W...,CT.angio,Chest,...,[CT],[dg.MD1R/a27cff27-dfd7-43bb-83a0-72e1bb4f56a8],[ct_series_file],[],[],[],[],[],/programs/TCIA/projects/RICORD_1a,NaN
9,4f8e95a0-f934-49ab-84c3-f36f6e78ef3d,TCIA-RICORD_1a,1.2.826.0.1.3680043.10.474.440808.1895,[440808-000011],65,87279-6,NaN,CT Chest for screening,CT,Chest,...,[CT],[dg.MD1R/0a6f0415-8b63-4d66-8f09-3d36c226ff01],[ct_series_file],[],[],[],[],[],/programs/TCIA/projects/RICORD_1a,[CHEST]


### Create a download manifest for the DICOM header files
---
Using the results of our query, we can now access imaging study data for our selected cohort:
* CT images using `object_id`,
* **Only the DICOM headers** for the CT images using `dicom_header_file_guid`.

In [9]:
imaging_studies_df[['object_id','dicom_header_file_guid']]

,object_id,dicom_header_file_guid
0,"[dg.MD1R/64a4b281-b1a9-4c05-b4a8-0597e8ca0abb,...","[dg.MD1R/24b4eb04-48bd-47ca-802d-4c7441318e40,..."
1,"[dg.MD1R/17e18624-48f3-481c-8eb5-ce662398ebfd,...","[dg.MD1R/bb1ea127-bc16-4d5e-8766-9ca91d313ef3,..."
2,[dg.MD1R/9698e6f9-9ec2-4811-b33a-edc43ae21f32],[dg.MD1R/7f9d9a20-86f5-4f7b-9395-b1fedfc3f210]
3,[dg.MD1R/d7a0311a-edb5-45a6-ae02-db500da40e1f],[dg.MD1R/0890903c-2078-4d38-8d1b-b9c6c1a6cb61]
4,[dg.MD1R/ed3fbd82-4075-448b-a55e-6bb27d83f940],[dg.MD1R/7521005c-79b2-4ce3-b157-e65a880244ad]
5,[dg.MD1R/6ed2e60c-0095-412a-a760-2a5841c18d14],[dg.MD1R/a47f1e42-478d-4687-bbc9-04286ad82f18]
6,"[dg.MD1R/4d067c79-d370-466b-9962-d130d58e9237,...","[dg.MD1R/4703d586-75d9-464a-a1fa-89e259128b63,..."
7,[dg.MD1R/2f89bee0-1b11-4ebb-898e-1188f596bedd],[dg.MD1R/385ac79c-3d20-40c8-8cec-e112c372ca1f]
8,[dg.MD1R/bbbbd15c-e39a-4c6d-a261-f995472284cf],[dg.MD1R/a27cff27-dfd7-43bb-83a0-72e1bb4f56a8]
9,[dg.MD1R/8becd54a-c5b6-4607-b7e1-323324497e29],[dg.MD1R/0a6f0415-8b63-4d66-8f09-3d36c226ff01]


In [10]:
## Flatten the column of header GUID lists for each imaging study into a single list of GUIDs
#list(imaging_studies_df['dicom_header_file_guid']))
header_guids = imaging_studies_df['dicom_header_file_guid'].apply(list).sum()
print(f"Creating a file download manifest for {len(header_guids)} DICOM header files for {len(imaging_studies_df)} imaging studies.")

Creating a file download manifest for 51 DICOM header files for 21 imaging studies.


In [11]:
## Define / Use a function to write the manifest file

def write_manifest(guids, filename="DICOM_header_file_manifest.json", output_dir="."):
        manifest_file = f"{output_dir}/{filename}"
        with open(manifest_file, "w") as manifest:
            manifest.write("[\n  {\n")
            count = 0
            for guid in guids:
                count += 1
                file_line = '    "object_id": "{}"\n'.format(guid)
                manifest.write(file_line)
                if count == len(guids):
                    manifest.write("  }]")
                else:
                    manifest.write("  },\n  {\n")
        print(f"File download manifest written to:\n\t{manifest_file}")
        return manifest_file

manifest_file = write_manifest(header_guids)

File download manifest written to:
	./DICOM_header_file_manifest.json


## 4) Access DICOM header files using their object_id / data GUID (globally unique identifiers)
---
Now that we have a list of DICOM header file GUIDs we want to download, we can use either the gen3-client or the gen3 SDK to download the files. 

For instructions on how to install and use the gen3-client, please see [the MIDRC quick-start guide](https://data.midrc.org/dashboard/Public/documentation/Gen3_MIDRC_GetStarted.pdf), which can be found linked here and in the MIDRC data portal header as "Get Started".

Below we use the gen3 SDK command `gen3 drs-pull object` which is [documented in detail here](https://github.com/uc-cdis/gen3sdk-python/blob/master/docs/howto/drsDownloading.md).

### Use the Gen3 SDK command `gen3 drs-pull object` to download an individual file

In [12]:
## Make a new directory in Colab /content dir for downloaded files
## Note: if this command is run in Google Colab, this will not alter any local directories
os.system("rm -r dicom_header_downloads")
os.system("mkdir -p dicom_header_downloads")


rm: dicom_header_downloads: No such file or directory


0

In [18]:
# ## We can use a simple loop to download all files and keep track of successes and failures
# ## Here we will only download one image to save time for demo purposes
# oid_num = 1
# success,failure,other=[],[],[]
# count,total = 0,len(object_ids[0:oid_num])
# for object_id in object_ids[0:oid_num]:
#     count+=1
#     cmd = "gen3 --auth {} --endpoint data.midrc.org drs-pull object {} --output-dir dicom_header_downloads".format(cred,object_id)
#     stout = subprocess.run(cmd, shell=True, capture_output=True)
#     if not stout.stdout:
#         raise Exception(f"gen3 sdk failure: {stout.stderr}")
#     #print("Progress ({}/{}): {}".format(count,total,stout.stdout))
#     print("Progress ({}/{}): {}".format(count,total,stout.stdout.decode("utf-8")))
#     if "failed" in str(stout.stdout):
#         failure.append(object_id)
#     elif "successfully" in str(stout.stdout):
#         success.append(object_id)
#     else:
#         other.append(object_id)


In [13]:
## Configure gen3-client

cmd = f"gen3-client configure --profile=midrcstaging --apiendpoint={sapi} --cred={scred}"
stout = subprocess.run(cmd, shell=True, capture_output=True)
if not stout.stderr:
    raise Exception(f"gen3 sdk failure: {stout.stderr}")
#print("Progress ({}/{}): {}".format(count,total,stout.stdout))
print(f"{stout.stderr.decode('utf-8')}")

2026/04/08 12:25:04 A new version of gen3-client is available! The latest version is 2026.4.0. You are using version 2025.09
2026/04/08 12:25:04 Please download the latest gen3-client release from https://github.com/uc-cdis/cdis-data-client/releases/latest
2026/04/08 12:25:05 Profile 'midrcstaging' has been configured successfully.



In [14]:
## We can use a simple loop to download all files and keep track of successes and failures
## Here we will only download one image to save time for demo purposes
download_path = 'dicom_header_downloads'
cmd = f"gen3-client download-multiple --profile=midrcstaging --manifest={manifest_file} --download-path={download_path} --numparallel=3 --no-prompt"
stout = subprocess.run(cmd, shell=True, capture_output=True)
if not stout.stderr:
    raise Exception(f"gen3 sdk failure: {stout.stderr}")
print(f"{stout.stderr.decode('utf-8')}")
# if "successfully" in str(stout.stdout):
#     success.append(object_id)
# else:
#     failure.append(object_id)


2026/04/08 12:25:12 A new version of gen3-client is available! The latest version is 2026.4.0. You are using version 2025.09
2026/04/08 12:25:12 Please download the latest gen3-client release from https://github.com/uc-cdis/cdis-data-client/releases/latest
2026/04/08 12:25:12 Reading manifest...
2026/04/08 12:25:12 Total number of objects in manifest: 51
2026/04/08 12:25:12 Preparing file info for each file, please wait...
2026/04/08 12:25:22 File info prepared successfully
2026/04/08 12:25:49 51 files downloaded.



In [ ]:
# Get a list of all downloaded .dcm files
header_zips = glob.glob(pathname=f'{download_path}/**/*_headers.zip',recursive=True,)
len(header_zips)

51

In [22]:
zip = header_zips[0]

In [25]:
print(zip)
print(download_path)

dicom_header_downloads/TCIA-RICORD_1a/419639-001596/1.2.826.0.1.3680043.10.474.419639.288267420128264959058930883141/1.2.826.0.1.3680043.10.474.419639.217659038521342318971586744906_headers.zip
dicom_header_downloads


In [23]:
study_dir = Path(zip).parent.absolute()
study_dir

PosixPath('/Users/qbatcheller/Desktop/dicom_header_downloads/TCIA-RICORD_1a/419639-001596/1.2.826.0.1.3680043.10.474.419639.288267420128264959058930883141')

In [34]:
for i in range(0,len(header_zips)):
    zip = header_zips[i]
    study_dir = Path(zip).parent.absolute()
    series_uid = Path(zip).stem.replace('_headers','')    
    print(f"({i+1}/{len(header_zips)}): {series_uid}")
    dcms = sorted(glob.glob(f"{study_dir}/{series_uid}/*.dcm"))
    if len(dcms) == 0: # unzip if not already unzipped
        print(f"({i+1}/{len(header_zips)}): Unzipping {series_uid}")
        os.system(f"unzip -q -o {zip} -d {study_dir}")
        

(1/51): 1.2.826.0.1.3680043.10.474.419639.217659038521342318971586744906
(2/51): 1.2.826.0.1.3680043.10.474.419639.222779076385397231259911522028
(3/51): 1.2.826.0.1.3680043.10.474.419639.248789328460452579567145413428
(4/51): 1.2.826.0.1.3680043.10.474.419639.433800011920523157764565727810
(5/51): 1.2.826.0.1.3680043.10.474.419639.148938292763592847048276595683
(6/51): 1.2.826.0.1.3680043.10.474.440808.1783
(7/51): 1.2.826.0.1.3680043.10.474.440808.1139
(8/51): 1.2.826.0.1.3680043.10.474.419639.124425916395023519732300401216
(9/51): 1.2.826.0.1.3680043.10.474.419639.281544670255145683323051531104
(10/51): 1.2.826.0.1.3680043.10.474.419639.371711514413844592850563324727
(11/51): 1.2.826.0.1.3680043.10.474.419639.871239758942100959793886583326
(12/51): 1.2.826.0.1.3680043.10.474.419639.324183195092595788008193224258
(13/51): 1.2.826.0.1.3680043.10.474.419639.138286829556512499358570895511
(14/51): 1.2.826.0.1.3680043.10.474.419639.544820382826068399575239149240
(15/51): 1.2.826.0.1.3680

In [35]:
series_uid

'1.2.826.0.1.3680043.10.474.440808.3023'

### Explore the DICOM Headers
---
Note that some of the files may contain compressed pixel data that require other packages to view; so, for this demo we'll simply skip over those using the following loop.

In [36]:
ds = pydicom.dcmread(dcms[0],force=True)
display(ds)

Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 180
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: CT Image Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.2.826.0.1.3680043.10.474.440808.3021
(0002,0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002,0012) Implementation Class UID            UI: 1.3.6.1.4.1.22213.1.143
(0002,0013) Implementation Version Name         SH: '0.5'
(0002,0016) Source Application Entity Title     AE: 'POSDA'
-------------------------------------------------
(0008,0005) Specific Character Set              CS: 'ISO_IR 100'
(0008,0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY', 'AXIAL']
(0008,0016) SOP Class UID                       UI: CT Image Storage
(0008,0018) SOP Instance UID                    UI: 1.2.826.0.1.3680043.10.474.440808.3021
(0008,0020) Study Date                

In [37]:
# Access individual elements
display(ds.file_meta)
display(ds.ImageType)
display(ds[0x0008, 0x0016])


(0002,0000) File Meta Information Group Length  UL: 180
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: CT Image Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.2.826.0.1.3680043.10.474.440808.3021
(0002,0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002,0012) Implementation Class UID            UI: 1.3.6.1.4.1.22213.1.143
(0002,0013) Implementation Version Name         SH: '0.5'
(0002,0016) Source Application Entity Title     AE: 'POSDA'

['ORIGINAL', 'PRIMARY', 'AXIAL']

(0008,0016) SOP Class UID                       UI: CT Image Storage

In [40]:
# View the dicom metadata for all files as a DataFrame
dfs = []
for image_file in dcms:
    ds = pydicom.dcmread(image_file)
    df = pd.DataFrame(ds.values())
    df[0] = df[0].apply(lambda x: pydicom.dataelem.convert_raw_data_element(x) if isinstance(x, pydicom.dataelem.RawDataElement) else x)
    df['name'] = df[0].apply(lambda x: x.name)
    df['value'] = df[0].apply(lambda x: x.value)
    df = df[['name', 'value']]
    df = df.set_index('name').T.reset_index(drop=True)
    df['filename'] = image_file
    #df.drop(columns=['Pixel Data'],inplace=True) # drop the pixel data as it's too large and nonsensical to store in a DataFrame
    dfs.append(df)

In [41]:
# Make a master dataframe for all images using only headers in all dataframes
headers = list(set.intersection(*map(set,dfs)))
df = pd.concat([df[headers] for df in dfs])
df.set_index('filename',inplace=True)


In [42]:
display(df)

name,Content Date,Bits Allocated,Image Type,Contrast/Bolus Route,Spacing Between Slices,Rows,Study Time,Acquisition Time,Series Time,Pixel Padding Value,...,Pixel Representation,Table Feed per Rotation,Manufacturer,Accession Number,Series Instance UID,Columns,Patient Position,Rescale Slope,Rescale Intercept,Samples per Pixel
filename,,,,,,,,,,,,,,,,,,,,,
/Users/qbatcheller/Desktop/dicom_header_downloads/TCIA-RICORD_1a/440808-000005/1.2.826.0.1.3680043.10.474.440808.3022/1.2.826.0.1.3680043.10.474.440808.3023/1-001_1.2.826.0.1.3680043.10.474.440808.3021_header.dcm,20041203,16,"[ORIGINAL, PRIMARY, AXIAL]",IV,1.25,512,135506,140001.297677,135940,-2000,...,1,39.375,,,1.2.826.0.1.3680043.10.474.440808.3023,512,FFS,1.0,-1024.0,1
/Users/qbatcheller/Desktop/dicom_header_downloads/TCIA-RICORD_1a/440808-000005/1.2.826.0.1.3680043.10.474.440808.3022/1.2.826.0.1.3680043.10.474.440808.3023/1-002_1.2.826.0.1.3680043.10.474.440808.3047_header.dcm,20041203,16,"[ORIGINAL, PRIMARY, AXIAL]",IV,1.25,512,135506,140001.297677,135940,-2000,...,1,39.375,,,1.2.826.0.1.3680043.10.474.440808.3023,512,FFS,1.0,-1024.0,1
/Users/qbatcheller/Desktop/dicom_header_downloads/TCIA-RICORD_1a/440808-000005/1.2.826.0.1.3680043.10.474.440808.3022/1.2.826.0.1.3680043.10.474.440808.3023/1-003_1.2.826.0.1.3680043.10.474.440808.3058_header.dcm,20041203,16,"[ORIGINAL, PRIMARY, AXIAL]",IV,1.25,512,135506,140001.297677,135940,-2000,...,1,39.375,,,1.2.826.0.1.3680043.10.474.440808.3023,512,FFS,1.0,-1024.0,1
/Users/qbatcheller/Desktop/dicom_header_downloads/TCIA-RICORD_1a/440808-000005/1.2.826.0.1.3680043.10.474.440808.3022/1.2.826.0.1.3680043.10.474.440808.3023/1-004_1.2.826.0.1.3680043.10.474.440808.3069_header.dcm,20041203,16,"[ORIGINAL, PRIMARY, AXIAL]",IV,1.25,512,135506,140001.297677,135940,-2000,...,1,39.375,,,1.2.826.0.1.3680043.10.474.440808.3023,512,FFS,1.0,-1024.0,1
/Users/qbatcheller/Desktop/dicom_header_downloads/TCIA-RICORD_1a/440808-000005/1.2.826.0.1.3680043.10.474.440808.3022/1.2.826.0.1.3680043.10.474.440808.3023/1-005_1.2.826.0.1.3680043.10.474.440808.3080_header.dcm,20041203,16,"[ORIGINAL, PRIMARY, AXIAL]",IV,1.25,512,135506,140001.297677,135940,-2000,...,1,39.375,,,1.2.826.0.1.3680043.10.474.440808.3023,512,FFS,1.0,-1024.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
/Users/qbatcheller/Desktop/dicom_header_downloads/TCIA-RICORD_1a/440808-000005/1.2.826.0.1.3680043.10.474.440808.3022/1.2.826.0.1.3680043.10.474.440808.3023/1-107_1.2.826.0.1.3680043.10.474.440808.3033_header.dcm,20041203,16,"[ORIGINAL, PRIMARY, AXIAL]",IV,1.25,512,135506,140001.297677,135940,-2000,...,1,39.375,,,1.2.826.0.1.3680043.10.474.440808.3023,512,FFS,1.0,-1024.0,1
/Users/qbatcheller/Desktop/dicom_header_downloads/TCIA-RICORD_1a/440808-000005/1.2.826.0.1.3680043.10.474.440808.3022/1.2.826.0.1.3680043.10.474.440808.3023/1-108_1.2.826.0.1.3680043.10.474.440808.3034_header.dcm,20041203,16,"[ORIGINAL, PRIMARY, AXIAL]",IV,1.25,512,135506,140001.297677,135940,-2000,...,1,39.375,,,1.2.826.0.1.3680043.10.474.440808.3023,512,FFS,1.0,-1024.0,1
/Users/qbatcheller/Desktop/dicom_header_downloads/TCIA-RICORD_1a/440808-000005/1.2.826.0.1.3680043.10.474.440808.3022/1.2.826.0.1.3680043.10.474.440808.3023/1-109_1.2.826.0.1.3680043.10.474.440808.3035_header.dcm,20041203,16,"[ORIGINAL, PRIMARY, AXIAL]",IV,1.25,512,135506,140001.297677,135940,-2000,...,1,39.375,,,1.2.826.0.1.3680043.10.474.440808.3023,512,FFS,1.0,-1024.0,1


In [43]:
## Export the file metadata as a TSV file
filename = "MIDRC_DICOM_metadata.tsv"
df.to_csv(filename, sep='\t')
